## Build Your First Semantic Cache

In [2]:
import pandas as pd
import numpy as np
import time

from cache.faq_data_container import FAQDataContainer

faq_data = FAQDataContainer()
faq_df = faq_data.faq_df
test_df = faq_data.test_df

Loaded 8 FAQ entries
Loaded 80 test queries


In [3]:
faq_df.head().style

,id,question,answer
0,0,How do I get a refund?,"To request a refund, visit your orders page and select **Request Refund**. Refunds are processed within 3-5 business days."
1,1,Can I reset my password?,Click **Forgot Password** on the login page and follow the email instructions. Check your spam folder if you don't see the email.
2,2,Where is my order?,Use the tracking link sent to your email after shipping. Orders typically arrive within 2-7 business days depending on your location.
3,3,How long is the warranty?,All electronic products include a 12-month warranty from the purchase date. Extended warranties are available for purchase.
4,4,Do you ship internationally?,"Yes, we ship to over 50 countries worldwide. International shipping fees and delivery times vary by destination."


### Create Embeddings for Semantic Search

In [ ]:
import sys
!{sys.executable} -m pip install "transformers>=4.41,<5.0"

In [8]:
import sys
!{sys.executable} -m pip install "numpy<2.0"


  Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl (20.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4


In [4]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-mpnet-base-v2")

faq_embeddings = encoder.encode(faq_df["question"].tolist())

print(f"Sample (first 10 dimensions): {faq_embeddings[0][:10]}")

/Users/I572648/Library/CloudStorage/OneDrive-SAPSE/Desktop/exp-gen/exp-gen/agent-learnings/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sample (first 10 dimensions): [ 0.0274638   0.04248317 -0.02679517  0.03786725 -0.03876132 -0.00109465
 -0.01158366  0.03070287 -0.00653772 -0.01955778]


### Implement semantic search

In [5]:
def cosine_dist(a: np.array, b: np.array):
    """Compute cosine distance between two sets of vectors."""
    a_norm = np.linalg.norm(a, axis=1)
    b_norm = np.linalg.norm(b) if b.ndim == 1 else np.linalg.norm(b, axis=1)
    sim = np.dot(a, b) / (a_norm * b_norm)
    return 1 - sim


def semantic_search(query: str) -> tuple:
    """Find the most similar FAQ question to the query."""
    query_embedding = encoder.encode([query])[0]

    distances = cosine_dist(faq_embeddings, query_embedding)

    # Find the most similar question (lowest distance)
    best_idx = int(np.argmin(distances))
    best_distance = distances[best_idx]

    return best_idx, best_distance

In [6]:
idx, distance = semantic_search(
    "How long will it take to get a refund for my order?"
)

print(f"Most similar FAQ: {faq_df.iloc[idx]['question']}")
print(f"Answer: {faq_df.iloc[idx]['answer']}")
print(f"Cosine distance: {distance:.3f}")

Most similar FAQ: How do I get a refund?
Answer: To request a refund, visit your orders page and select **Request Refund**. Refunds are processed within 3-5 business days.
Cosine distance: 0.331


### Build a Simple Semantic Cache

In [7]:
def check_cache(query: str, distance_threshold: float = 0.3):
    """
    Semantic cache lookup for previously asked questions.
    Returns a dictionary with answer if hit, None if miss.
    """
    idx, distance = semantic_search(query)

    if distance <= distance_threshold:
        return {
            "prompt": faq_df.iloc[idx]["question"],
            "response": faq_df.iloc[idx]["answer"],
            "vector_distance": float(distance),
        }

    return None

In [8]:
test_queries = [
    "Is it possible to get a refund?",
    "I want my money back",
    "What are your business hours?",  # Should miss
]

for query in test_queries:
    result = check_cache(query, distance_threshold=0.3)
    if result:
        print(f"✅ HIT: '{query}' -> {result['response'][:50]}...")
        print(f"   Distance: {result['vector_distance']:.3f}\n")
    else:
        print(f"❌ MISS: '{query}'\n")

✅ HIT: 'Is it possible to get a refund?' -> To request a refund, visit your orders page and se...
   Distance: 0.262

❌ MISS: 'I want my money back'

❌ MISS: 'What are your business hours?'



### Add entries to the cache

In [9]:
def add_to_cache(question: str, answer: str):
    """
    Add a new Q&A pair to our simple in-memory cache.
    Extends both the DataFrame and embeddings matrix.
    """
    global faq_df, faq_embeddings

    new_row = pd.DataFrame({"question": [question], "answer": [answer]})
    faq_df = pd.concat([faq_df, new_row], ignore_index=True)

    # Generate embedding for the new question
    new_embedding = encoder.encode([question])

    # Add to embeddings matrix
    faq_embeddings = np.vstack([faq_embeddings, new_embedding])

    print(f"✅ Added to cache: '{question}'")

In [10]:
print("Original cache size:", len(faq_df))

new_entries = [
    (
        "What are your business hours?",
        "We're open Monday-Friday 9 AM to 6 PM EST. Weekend support is available for urgent issues.",
    ),
    (
        "Do you have a mobile app?",
        "Yes! Our mobile app is available on both iOS and Android. Search for 'CustomerApp' in your app store.",
    ),
    (
        "How do I update my payment method?",
        "Go to Account Settings > Payment Methods to add, edit, or remove payment options.",
    ),
]

for question, answer in new_entries:
    add_to_cache(question, answer)

print(f"\nCache now has {len(faq_df)} total entries")

Original cache size: 8
✅ Added to cache: 'What are your business hours?'
✅ Added to cache: 'Do you have a mobile app?'
✅ Added to cache: 'How do I update my payment method?'

Cache now has 11 total entries


In [11]:
test_extended_queries = [
    "What time do you open?",  
    "Is there a phone app?", 
    "How can I change my payment method?",
]

for query in test_extended_queries:
    result = check_cache(query, distance_threshold=0.3)
    if result:
        print(f"✅ HIT: '{query}' -> {result['response'][:50]}...")
        print(f"   Distance: {result['vector_distance']:.3f}\n")
    else:
        print(f"❌ MISS: '{query}'\n")

✅ HIT: 'What time do you open?' -> We're open Monday-Friday 9 AM to 6 PM EST. Weekend...
   Distance: 0.289

✅ HIT: 'Is there a phone app?' -> Yes! Our mobile app is available on both iOS and A...
   Distance: 0.265

✅ HIT: 'How can I change my payment method?' -> Go to Account Settings > Payment Methods to add, e...
   Distance: 0.118



### Moving to Redis

In [12]:
import os

REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

In [27]:
import sys
!{sys.executable} -m pip install redis

In [28]:
import redis

try:
    r = redis.Redis.from_url(REDIS_URL)
    r.ping()
    print("✅ Redis is running and accessible")
except redis.ConnectionError:
    print("❌ Cannot connect to Redis")
    raise

✅ Redis is running and accessible


### Using a Cache-Optimized Embedding Model (langcache-embed-v1)
https://huggingface.co/redis/langcache-embed-v1

In [30]:
import sys
!{sys.executable} -m pip install redisvl

In [22]:
import torch
# torch.compile (Dynamo) doesn't support Python 3.12 in torch<2.3.
# Patch it to a no-op so ModernBertModel can be imported; inference still works.
torch.compile = lambda fn=None, **kwargs: (fn if fn is not None else lambda f: f)


In [23]:
from redisvl.utils.vectorize import HFTextVectorizer
from redisvl.extensions.cache.embeddings import EmbeddingsCache

langcache_embed = HFTextVectorizer(
    model="redis/langcache-embed-v1",
    cache=EmbeddingsCache(redis_client=r, ttl=3600)
)

### Create the Redis Semantic Cache

In [ ]:
from redisvl.extensions.cache.llm import SemanticCache

cache = SemanticCache(
    name="faq-cache",
    vectorizer=langcache_embed,
    redis_client=r,
    distance_threshold=0.3
)

### Load the Cache with FAQ Data

In [ ]:
for i in range(len(faq_df)):
    cache.store(
        prompt=faq_df.iloc[i]["question"],
        response=faq_df.iloc[i]["answer"]
    )

In [ ]:
result = cache.check("I need a refund for my purchase")
result

### Implement TTL (time-to-live) policy to keep cache fresh

In [ ]:
cache.set_ttl(86400)

## End-to-End LLM Example

In [ ]:
from cache.config import load_openai_key
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_openai_key()

MODEL_NAME = "gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.1,
    max_tokens=150,
)

In [ ]:
def get_llm_response(question: str) -> str:
    prompt = f"""
    You are a helpful customer support assistant. Answer this customer question concisely and professionally:
    
    Question: {question}
    
    Provide a helpful response in 1-2 sentences. If you don't have specific information, give a general helpful response.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

In [ ]:
from cache.evals import PerfEval

perf_eval = PerfEval()

test_questions = [
    "How can I get my money back?",
    "I want a refund please",
    "What's your return policy?",
    "I forgot my password",
    "Can you help me reset my password?",
    "What are your shipping costs?",
    "Do you offer installation services?",
    "Can I schedule a phone call with support?",
    "How do I cancel my subscription?",
    "How much does shipping cost?",
    "I need to cancel my account",
]

perf_eval.set_total_queries(len(test_questions))

In [ ]:
with perf_eval:
    for i, question in enumerate(test_questions, 1):
        print(f"\n[{i}] Question: '{question}'")

        perf_eval.start()

        if cached_result := cache.check(question):
            # Cache HIT
            perf_eval.tick("cache_hit")
            print(
                f"    ✅ CACHE HIT (distance: {cached_result[0]['vector_distance']:.3f})"
            )
            print(f"    📋 Cached question: {cached_result[0]['prompt'][:80]}...")
            print(f"    📋 Cached response: {cached_result[0]['response'][:80]}...")
        else:
            # Cache MISS - call LLM
            perf_eval.tick("cache_miss")  # Time for cache check
            print(f"    ❌ CACHE MISS")
            print(f"    🤖 Calling LLM... ", end="")

            # Call LLM and track the call
            perf_eval.start()
            llm_response = get_llm_response(question)
            perf_eval.tick("llm_call")
            perf_eval.record_llm_call(MODEL_NAME, question, llm_response)
            print(f"    💬 LLM response: {llm_response[:80]}...")
            cache.store(prompt=question, response=llm_response)

In [ ]:
np.mean(perf_eval.durations_by_label['cache_hit'])

In [ ]:
np.mean(perf_eval.durations_by_label['llm_call'])